# Nível 1 — Dados e primeira análise com LLM

**Desafio Técnico — Estágio em Engenharia de Inteligência Artificial**

Objetivo: tratar a base pequena, normalizar valores para BRL, produzir agregações, implementar e validar as duas regras determinísticas e preparar uma análise estruturada com LLM.

> Princípio de arquitetura: **pandas calcula; a LLM interpreta**. A LLM não é usada para soma, mediana, contagem nem comparação com limites.


## 1. Imports e carregamento

O notebook detecta automaticamente se está sendo executado a partir da raiz do repositório ou da pasta `nivel_1/`.


In [1]:
from pathlib import Path
import json
import os
import time

import numpy as np
import pandas as pd
from dotenv import load_dotenv
from IPython.display import display

pd.set_option("display.max_columns", 30)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

BASE_DIR = Path.cwd()
if not (BASE_DIR / "dados").exists():
    BASE_DIR = BASE_DIR.parent

load_dotenv(BASE_DIR / ".env")

DADOS_PATH = BASE_DIR / "dados" / "dados_nivel_1.json"
OUTPUTS_DIR = BASE_DIR / "outputs"
OUTPUTS_DIR.mkdir(exist_ok=True)

with open(DADOS_PATH, encoding="utf-8") as f:
    payload = json.load(f)

taxa_cambio = float(payload["taxa_cambio_usd_brl"])
df_raw = pd.DataFrame(payload["operacoes"])

print(f"Taxa USD/BRL fornecida: {taxa_cambio}")
print(f"Registros recebidos: {len(df_raw)}")
print(f"Clientes distintos: {df_raw['cliente_id'].nunique()}")
display(df_raw.head())

Taxa USD/BRL fornecida: 5.4
Registros recebidos: 20
Clientes distintos: 6


,id,cliente_id,data,valor,moeda,canal,tipo,contraparte,observacao
0,OP-0001,CLI-A-1,2026-03-09,18100,BRL,pix,transferencia_enviada,Alfa Comercio LTDA,
1,OP-0002,CLI-A-1,2026-03-09,17300,BRL,pix,transferencia_enviada,Alfa Comercio LTDA,
2,OP-0003,CLI-A-1,2026-03-09,18800,BRL,ted,transferencia_enviada,Beta Servicos ME,
3,OP-0004,CLI-A-1,2026-03-21,3300,BRL,boleto,pagamento,Gama Distribuidora,
4,OP-0005,CLI-A-2,2026-03-14,25900,BRL,ted,transferencia_enviada,Delta Transportes,


## 2. Diagnóstico e tratamento de qualidade

### Problemas encontrados

1. **ID duplicado:** `OP-0007` aparece duas vezes com conteúdo idêntico.  
   **Tratamento:** manter a primeira ocorrência e remover a cópia. Como `id` representa uma operação, contar a duplicata distorceria volume, mediana, contagens e regras.

2. **Data ausente:** a operação `OP-0017` possui `data = null` e observação de que a data não foi capturada.  
   **Tratamento:** manter a operação no histórico e nas análises que não dependem de data; representar a data como `NaT`. Ela é excluída apenas da Regra 1, que exige agrupamento por dia. Não imputo uma data porque isso inventaria evidência.

3. **Moedas diferentes:** há BRL e USD.  
   **Tratamento:** criar `valor_brl` usando a taxa fixa fornecida no arquivo. O campo original é preservado para rastreabilidade.

Essas decisões preservam o máximo de informação sem contaminar os cálculos.


In [2]:
print("Duplicatas por ID:")
display(df_raw[df_raw.duplicated("id", keep=False)].sort_values("id"))

print("\nValores ausentes por coluna:")
display(df_raw.isna().sum().to_frame("qtd_ausentes"))

print("\nMoedas encontradas:")
display(df_raw["moeda"].value_counts().to_frame("qtd"))

Duplicatas por ID:


,id,cliente_id,data,valor,moeda,canal,tipo,contraparte,observacao
6,OP-0007,CLI-A-3,2026-03-05,17200,BRL,pix,transferencia_enviada,Epsilon Consultoria,
9,OP-0007,CLI-A-3,2026-03-05,17200,BRL,pix,transferencia_enviada,Epsilon Consultoria,



Valores ausentes por coluna:


,qtd_ausentes
id,0
cliente_id,0
data,1
valor,0
moeda,0
canal,0
tipo,0
contraparte,0
observacao,0



Moedas encontradas:


,qtd
moeda,
BRL,19
USD,1


In [3]:
df = df_raw.drop_duplicates(subset=["id"], keep="first").copy()
df["data"] = pd.to_datetime(df["data"], errors="coerce")
df["data_ausente"] = df["data"].isna()

df["valor_brl"] = np.where(
    df["moeda"].str.upper().eq("USD"),
    df["valor"].astype(float) * taxa_cambio,
    df["valor"].astype(float),
)

print(f"Registros após deduplicação: {len(df)}")
print(f"Datas ausentes preservadas: {df['data'].isna().sum()}")
display(df.loc[df["moeda"].eq("USD"), ["id", "cliente_id", "valor", "moeda", "valor_brl"]])

Registros após deduplicação: 19
Datas ausentes preservadas: 1


,id,cliente_id,valor,moeda,valor_brl
13,OP-0013,CLI-A-4,12000,USD,"64,800.00"


## 3. Agregações pedidas

- volume total transacionado por cliente, já em BRL;
- quantidade de operações por canal.


In [4]:
volume_por_cliente = (
    df.groupby("cliente_id", as_index=False)["valor_brl"]
      .sum()
      .rename(columns={"valor_brl": "volume_total_brl"})
      .sort_values("volume_total_brl", ascending=False)
)

operacoes_por_canal = (
    df.groupby("canal", as_index=False)["id"]
      .count()
      .rename(columns={"id": "qtd_operacoes"})
      .sort_values("qtd_operacoes", ascending=False)
)

print("Volume total por cliente:")
display(volume_por_cliente)

print("Quantidade de operações por canal:")
display(operacoes_por_canal)

Volume total por cliente:


,cliente_id,volume_total_brl
3,CLI-A-4,"79,500.00"
0,CLI-A-1,"57,500.00"
1,CLI-A-2,"52,900.00"
2,CLI-A-3,"48,500.00"
4,CLI-A-5,"16,900.00"
5,CLI-A-6,"10,200.00"


Quantidade de operações por canal:


,canal,qtd_operacoes
3,pix,8
4,ted,5
0,boleto,3
1,cartao,2
2,especie,1


## 4. Regra 1 — Fracionamento

Sinalizar um **cliente em uma mesma data** quando:

- houver 3 ou mais operações;
- a soma ultrapassar R$ 50.000;
- nenhuma operação isolada atingir R$ 20.000.

A regra é calculada por grupo `cliente_id + data`. A flag é depois propagada às operações do grupo para facilitar auditoria.


In [5]:
resumo_dia = (
    df.dropna(subset=["data"])
      .groupby(["cliente_id", "data"], as_index=False)
      .agg(
          qtd_operacoes_dia=("id", "size"),
          volume_dia_brl=("valor_brl", "sum"),
          maior_operacao_dia_brl=("valor_brl", "max"),
      )
)

resumo_dia["flag_fracionamento_grupo"] = (
    (resumo_dia["qtd_operacoes_dia"] >= 3)
    & (resumo_dia["volume_dia_brl"] > 50_000)
    & (resumo_dia["maior_operacao_dia_brl"] < 20_000)
)

df = df.merge(
    resumo_dia,
    on=["cliente_id", "data"],
    how="left"
)
df["flag_fracionamento"] = df["flag_fracionamento_grupo"].eq(True)

display(resumo_dia[resumo_dia["flag_fracionamento_grupo"]])

,cliente_id,data,qtd_operacoes_dia,volume_dia_brl,maior_operacao_dia_brl,flag_fracionamento_grupo
0,CLI-A-1,2026-03-09,3,"54,200.00","18,800.00",True


## 5. Validação explícita da Regra 1

O enunciado pede um **caso positivo** e um **caso parecido que não se enquadra**.

- `CLI-A-1`, 09/03/2026: 3 operações, soma R$ 54.200, maior operação R$ 18.800 → **capturado**.
- `CLI-A-2`, 14/03/2026: soma R$ 52.900, porém são apenas 2 operações e ambas passam de R$ 20 mil → **não capturado**.

Isso testa simultaneamente os três critérios da regra.


In [6]:
casos_validacao = pd.DataFrame({
    "cliente_id": ["CLI-A-1", "CLI-A-2"],
    "data": pd.to_datetime(["2026-03-09", "2026-03-14"]),
    "resultado_esperado": [True, False],
})

validacao_r1 = casos_validacao.merge(
    resumo_dia,
    on=["cliente_id", "data"],
    how="left",
)

validacao_r1["validacao_ok"] = (
    validacao_r1["flag_fracionamento_grupo"]
    == validacao_r1["resultado_esperado"]
)

display(validacao_r1[
    ["cliente_id", "data", "qtd_operacoes_dia", "volume_dia_brl",
     "maior_operacao_dia_brl", "flag_fracionamento_grupo",
     "resultado_esperado", "validacao_ok"]
])

assert validacao_r1["validacao_ok"].all()
print("Validação da Regra 1: OK")

,cliente_id,data,qtd_operacoes_dia,volume_dia_brl,maior_operacao_dia_brl,flag_fracionamento_grupo,resultado_esperado,validacao_ok
0,CLI-A-1,2026-03-09,3,"54,200.00","18,800.00",True,True,True
1,CLI-A-2,2026-03-14,2,"52,900.00","27,000.00",False,False,True


Validação da Regra 1: OK


## 6. Regra 2 — Valor atípico

Sinalizar uma operação quando:

- o cliente tiver 4 ou mais operações;
- `valor_brl > 5 × mediana_cliente_brl`.

A mediana e o limite são calculados por pandas.


In [7]:
stats_cliente = (
    df.groupby("cliente_id", as_index=False)
      .agg(
          qtd_operacoes_cliente=("id", "size"),
          mediana_cliente_brl=("valor_brl", "median"),
      )
)

df = df.merge(stats_cliente, on="cliente_id", how="left")
df["limite_atipico_brl"] = 5 * df["mediana_cliente_brl"]
df["flag_valor_atipico"] = (
    (df["qtd_operacoes_cliente"] >= 4)
    & (df["valor_brl"] > df["limite_atipico_brl"])
)

atipicas = df[df["flag_valor_atipico"]].copy()
display(atipicas[
    ["id", "cliente_id", "valor", "moeda", "valor_brl",
     "qtd_operacoes_cliente", "mediana_cliente_brl",
     "limite_atipico_brl", "flag_valor_atipico"]
])

,id,cliente_id,valor,moeda,valor_brl,qtd_operacoes_cliente,mediana_cliente_brl,limite_atipico_brl,flag_valor_atipico
12,OP-0013,CLI-A-4,12000,USD,"64,800.00",4,"5,450.00","27,250.00",True


### Leitura do resultado

A operação `OP-0013` do `CLI-A-4` é de **US$ 12.000**. Com a taxa fixa 5,4, o valor normalizado é **R$ 64.800**.

Para esse cliente:
- mediana = **R$ 5.450**;
- 5 × mediana = **R$ 27.250**;
- R$ 64.800 > R$ 27.250.

Logo a operação é corretamente sinalizada pela Regra 2.


## 7. DataFrame final com flags

As duas flags são adicionadas ao DataFrame tratado. O arquivo é salvo em `outputs/` para auditabilidade.


In [8]:
colunas_exibicao = [
    "id", "cliente_id", "data", "valor", "moeda", "valor_brl", "canal", "tipo",
    "flag_fracionamento", "flag_valor_atipico"
]
display(df[colunas_exibicao])

df_export = df.copy()
df_export["data"] = df_export["data"].dt.strftime("%Y-%m-%d")
df_export.to_csv(OUTPUTS_DIR / "nivel_1_operacoes_tratadas.csv", index=False)
volume_por_cliente.to_csv(OUTPUTS_DIR / "nivel_1_volume_por_cliente.csv", index=False)
operacoes_por_canal.to_csv(OUTPUTS_DIR / "nivel_1_operacoes_por_canal.csv", index=False)

print("Outputs determinísticos salvos.")

,id,cliente_id,data,valor,moeda,valor_brl,canal,tipo,flag_fracionamento,flag_valor_atipico
0,OP-0001,CLI-A-1,2026-03-09,18100,BRL,"18,100.00",pix,transferencia_enviada,True,False
1,OP-0002,CLI-A-1,2026-03-09,17300,BRL,"17,300.00",pix,transferencia_enviada,True,False
2,OP-0003,CLI-A-1,2026-03-09,18800,BRL,"18,800.00",ted,transferencia_enviada,True,False
3,OP-0004,CLI-A-1,2026-03-21,3300,BRL,"3,300.00",boleto,pagamento,False,False
4,OP-0005,CLI-A-2,2026-03-14,25900,BRL,"25,900.00",ted,transferencia_enviada,False,False
5,OP-0006,CLI-A-2,2026-03-14,27000,BRL,"27,000.00",ted,transferencia_enviada,False,False
6,OP-0007,CLI-A-3,2026-03-05,17200,BRL,"17,200.00",pix,transferencia_enviada,False,False
7,OP-0008,CLI-A-3,2026-03-05,15200,BRL,"15,200.00",pix,transferencia_enviada,False,False
8,OP-0009,CLI-A-3,2026-03-05,16100,BRL,"16,100.00",pix,transferencia_enviada,False,False
9,OP-0010,CLI-A-4,2026-03-03,3800,BRL,"3,800.00",cartao,pagamento,False,False


Outputs determinísticos salvos.


# Parte B — Análise com LLM

Cliente escolhido: **CLI-A-4**, sinalizado pela Regra 2.

A LLM não receberá a tarefa de recalcular a regra. Ela receberá fatos já calculados e deverá apenas interpretar/redigir.

A saída é validada por Pydantic com:
- `nivel_risco`: baixo/médio/alto;
- `tipologia_suspeita`;
- `red_flags`;
- `justificativa`.

Também são medidos tokens e latência. Se a saída falhar na validação, há um retry corretivo.


In [9]:
cliente_escolhido = "CLI-A-4"
caso = df[df["cliente_id"].eq(cliente_escolhido)].copy()

fatos_llm = {
    "cliente_id": cliente_escolhido,
    "qtd_operacoes": int(len(caso)),
    "volume_total_brl": round(float(caso["valor_brl"].sum()), 2),
    "mediana_cliente_brl": round(float(caso["mediana_cliente_brl"].iloc[0]), 2),
    "alertas": {
        "fracionamento": bool(caso["flag_fracionamento"].any()),
        "operacoes_valor_atipico": caso.loc[
            caso["flag_valor_atipico"],
            ["id", "valor_brl", "limite_atipico_brl", "canal", "tipo", "contraparte"]
        ].to_dict(orient="records"),
    }
}
print(json.dumps(fatos_llm, ensure_ascii=False, indent=2, default=str))

{
  "cliente_id": "CLI-A-4",
  "qtd_operacoes": 4,
  "volume_total_brl": 79500.0,
  "mediana_cliente_brl": 5450.0,
  "alertas": {
    "fracionamento": false,
    "operacoes_valor_atipico": [
      {
        "id": "OP-0013",
        "valor_brl": 64800.00000000001,
        "limite_atipico_brl": 27250.0,
        "canal": "ted",
        "tipo": "transferencia_recebida",
        "contraparte": "Zeta Importacao"
      }
    ]
  }
}


## 8. Duas versões de prompt

**Prompt V1** é propositalmente mais curto e aberto.  
**Prompt V2** explicita papel, limites, proibição de recalcular e necessidade de basear a justificativa somente nas evidências.

Hipótese a validar após execução: V2 tende a ser mais auditável e menos propenso a extrapolar fatos.


In [10]:
PROMPT_V1 = f"""
Analise o cliente abaixo sob a ótica de prevenção à lavagem de dinheiro
e produza um parecer estruturado.

Fatos:
{json.dumps(fatos_llm, ensure_ascii=False, default=str)}
""".strip()

PROMPT_V2 = f"""
Atue como analista de Prevenção à Lavagem de Dinheiro.

Regras de trabalho:
1. Use SOMENTE os fatos fornecidos.
2. Não recalcule soma, mediana, contagem ou limites.
3. Não invente contexto cadastral, renda, setor ou intenção.
4. Um alerta determinístico é um indício para triagem, não prova de ilícito.
5. Explique quais evidências sustentam o nível de risco e a tipologia.

Fatos determinísticos já calculados por pandas:
{json.dumps(fatos_llm, ensure_ascii=False, default=str)}

Produza um parecer conciso e auditável.
""".strip()

print("PROMPT V1:\n", PROMPT_V1)
print("\n" + "="*80 + "\n")
print("PROMPT V2:\n", PROMPT_V2)

PROMPT V1:
 Analise o cliente abaixo sob a ótica de prevenção à lavagem de dinheiro
e produza um parecer estruturado.

Fatos:
{"cliente_id": "CLI-A-4", "qtd_operacoes": 4, "volume_total_brl": 79500.0, "mediana_cliente_brl": 5450.0, "alertas": {"fracionamento": false, "operacoes_valor_atipico": [{"id": "OP-0013", "valor_brl": 64800.00000000001, "limite_atipico_brl": 27250.0, "canal": "ted", "tipo": "transferencia_recebida", "contraparte": "Zeta Importacao"}]}}


PROMPT V2:
 Atue como analista de Prevenção à Lavagem de Dinheiro.

Regras de trabalho:
1. Use SOMENTE os fatos fornecidos.
2. Não recalcule soma, mediana, contagem ou limites.
3. Não invente contexto cadastral, renda, setor ou intenção.
4. Um alerta determinístico é um indício para triagem, não prova de ilícito.
5. Explique quais evidências sustentam o nível de risco e a tipologia.

Fatos determinísticos já calculados por pandas:
{"cliente_id": "CLI-A-4", "qtd_operacoes": 4, "volume_total_brl": 79500.0, "mediana_cliente_brl": 5

In [11]:
from typing import Literal
from pydantic import BaseModel, ValidationError

class ParecerPLD(BaseModel):
    nivel_risco: Literal["baixo", "médio", "alto"]
    tipologia_suspeita: str
    red_flags: list[str]
    justificativa: str

def chamar_gemini(prompt: str, max_retries: int = 1):
    """Chamada estruturada com medição de latência/tokens e retry de validação."""
    from google import genai
    from google.genai import types

    api_key = os.getenv("GEMINI_API_KEY", "").strip()
    if not api_key:
        raise RuntimeError("GEMINI_API_KEY não configurada.")

    model = os.getenv("GEMINI_MODEL", "gemini-2.5-flash-lite")
    client = genai.Client(api_key=api_key)

    erro_anterior = None
    for tentativa in range(max_retries + 1):
        prompt_atual = prompt
        if tentativa > 0:
            prompt_atual += (
                "\n\nA resposta anterior não validou. "
                "Retorne somente uma resposta compatível com o schema exigido."
            )

        t0 = time.perf_counter()
        response = client.models.generate_content(
            model=model,
            contents=prompt_atual,
            config=types.GenerateContentConfig(
                response_mime_type="application/json",
                response_schema=ParecerPLD,
            ),
        )
        latencia = time.perf_counter() - t0

        try:
            parecer = ParecerPLD.model_validate_json(response.text)
            usage = getattr(response, "usage_metadata", None)
            metricas = {
                "modelo": model,
                "latencia_s": round(latencia, 4),
                "tokens_entrada": getattr(usage, "prompt_token_count", None),
                "tokens_saida": getattr(usage, "candidates_token_count", None),
                "tentativa": tentativa + 1,
            }
            return parecer, metricas
        except ValidationError as e:
            erro_anterior = e

    raise RuntimeError(f"Resposta malformada após retry: {erro_anterior}")

print("Função de chamada estruturada pronta.")

Função de chamada estruturada pronta.


In [12]:
# Esta célula é segura para o repositório: sem chave, ela NÃO inventa uma execução.
resultados_prompts = {}

if not os.getenv("GEMINI_API_KEY", "").strip():
    print(
        "LLM NÃO EXECUTADA NESTE AMBIENTE: GEMINI_API_KEY ausente.\n"
        "Antes da entrega, configure .env/GEMINI_API_KEY e execute esta célula novamente.\n"
        "Tokens, latência e pareceres não serão simulados."
    )
else:
    for nome, prompt in {"v1": PROMPT_V1, "v2": PROMPT_V2}.items():
        parecer, metricas = chamar_gemini(prompt)
        resultados_prompts[nome] = {
            "parecer": parecer.model_dump(),
            "metricas": metricas,
        }
        print(f"\n{name if False else nome.upper()}")
        print(json.dumps(resultados_prompts[nome], ensure_ascii=False, indent=2))

    with open(OUTPUTS_DIR / "nivel_1_comparacao_prompts.json", "w", encoding="utf-8") as f:
        json.dump(resultados_prompts, f, ensure_ascii=False, indent=2)

LLM NÃO EXECUTADA NESTE AMBIENTE: GEMINI_API_KEY ausente.
Antes da entrega, configure .env/GEMINI_API_KEY e execute esta célula novamente.
Tokens, latência e pareceres não serão simulados.


## 9. Como comparar os prompts após a execução

Após gerar V1 e V2, a comparação deve observar:

- se a resposta respeitou os fatos sem inventar contexto;
- se a tipologia foi coerente com a evidência;
- se as red flags são específicas e rastreáveis;
- se a justificativa explica o nível de risco;
- diferença de tokens e latência.

**Critério esperado:** o Prompt V2 é preferível se produzir uma resposta mais fundamentada e restrita às evidências, mesmo que consuma alguns tokens a mais.

> Antes da entrega final, substituir este comentário por 2–4 linhas descrevendo o que de fato ocorreu nas duas execuções.
